In [ ]:
%load_ext autoreload
%autoreload 2

In [16]:
import pathlib
import tempfile
import subprocess

from llm_judge_helper import (
    read_llm_response_csv,
    get_table_info_with_descriptions,
    get_llm_response_comparison_score,
    parse_llm_output,
    prepare_csv_output,
    write_llm_output_to_csv,
)

In [ ]:
import dotenv

dotenv.load_dotenv()

In [10]:
csv1 = "output/llm_output_1744018433.4380157_parsed.csv"
csv2 = "output/llm_output_1744019521.8116157_parsed.csv"
testing_mode = False

In [ ]:
dict1 = read_llm_response_csv(csv1)
dict2 = read_llm_response_csv(csv2)
table_info_with_descriptions = get_table_info_with_descriptions(dict1, dict2)

In [12]:
if testing_mode:
    # Filter down to a subset of URNs for testing.
    test_urns = [
        "urn:li:dataset:(urn:li:dataPlatform:snowflake,edw_db.core.fct_settled_transaction,PROD)"
    ]

    table_info_with_descriptions_raw = table_info_with_descriptions.copy()
    table_info_with_descriptions = {
        ref: table_info_with_descriptions_raw[ref]
        for ref in table_info_with_descriptions_raw
        if ref.urn in test_urns
    }

In [ ]:
comparison_file = pathlib.Path(tempfile.mkdtemp()) / "comparison_scores.csv"

comparison_score_dict = get_llm_response_comparison_score(table_info_with_descriptions)
parsed_llm_output = {}
for urn in comparison_score_dict.keys():
    if comparison_score_dict.get(urn) is None:
        parsed_llm_output[urn] = None
    else:
        parsed_llm_output[urn] = parse_llm_output(comparison_score_dict[urn])
formatted_llm_output = prepare_csv_output(parsed_llm_output, dict1, dict2)
write_llm_output_to_csv(formatted_llm_output, comparison_file)

subprocess.check_call(["open", comparison_file])

In [ ]:
urn_idx = 0
instance_idx = 1
csv1_desc_idx = 2
csv2_desc_idx = 3
csv1_score_idx = 4
csv2_score_idx = 5
csv1_reasoning_idx = 6
csv2_reasoning_idx = 7
csv1_score_computation_logic_idx = 8
csv2_score_computation_logic_idx = 9

for row in formatted_llm_output:
    print("Urn: ", row[urn_idx])
    print("Instance: ", row[instance_idx])
    print("CSV1 Score: ", row[csv1_score_idx])
    print("CSV2 Score: ", row[csv2_score_idx])
    print("CSV1 Score Computation Logic: ", row[csv1_score_computation_logic_idx])
    print("CSV2 Score Computation Logic: ", row[csv2_score_computation_logic_idx])
    print("--------------------------------")